## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

In [ ]:
# tools inforamtion 
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = "your api key"

model = ChatGoogleGenerativeAI(model = "gemini-3-flash-preview")
response = model.invoke("How much an elephant drink water?")
response.content

[{'type': 'text',
  'text': 'An adult elephant typically drinks between **40 and 50 gallons (150 to 200 liters)** of water a day.\n\nHowever, this amount can change based on several factors:\n\n*   **Size and Species:** Larger African bull elephants may drink up to **55–60 gallons** in a single day, especially in hot weather.\n*   **Weather:** In very hot, dry conditions, an elephant can consume more than 60 gallons to stay hydrated.\n*   **The Trunk:** An elephant doesn’t drink through its trunk like a straw; it uses its trunk to suck up water and then squirts it into its mouth. A large elephant can hold about **2 to 3 gallons (8 to 12 liters)** of water in its trunk at one time.\n*   **Frequency:** If water is scarce, elephants can go two or three days without drinking, but when they finally find a water hole, they may drink massive amounts at once to compensate.\n\nBecause they require so much water, elephants are never usually found very far from a water source.',
  'extras': {'sig

### Calling all gemini model viva APIs.....

In [16]:
import google.generativeai as genai

for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

d:\Master-AI-Work\ai-master\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ashwi\AppData\Local\Temp\ipykernel_8816\3477461741.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [13]:
%pip install google-generativeai

  Using cached protobuf-7.35.0-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached proto_plus-1.28.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl.metadata (8.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.80.0-py3-none-any.whl.metadata (1.3 kB)
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.3 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 4.6 MB/s  0:00:00
Using cached googleapis_common_protos-1.75.0-py3-none-any.whl (300 kB)
   --------------------------------------

In [21]:
from langchain.tools import tool

# decorator to create a tool
@tool
def get_weather(location: str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

# binding my tools with LLM
model_with_tools = model.bind_tools([get_weather])

In [24]:
response = model_with_tools.invoke("what is the weather in SF?")
print(response.content)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


[]
Tool: get_weather
Args: {'location': 'San Francisco'}


## Tool Execution Loops

In [25]:
# Step 1: Model generates tools calls
messages = [{"role": "user", "content": "what's the weather in SF?"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message) # here message is just tool call

# Step 2: Execute tools and collect results
for tool_call in ai_message.tool_calls:
    # Execute the tool call and get results
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)

# The current weather in booston is 75 degrees and sunny.


[{'type': 'text', 'text': "It's currently sunny in San Francisco."}]


In [29]:
messages

[{'role': 'user', 'content': "what's the weather in SF?"},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "San Francisco"}'}, '__gemini_function_call_thought_signatures__': {'fc85df7a-455d-453e-a2cf-a6bfa2fabbee': 'Eo0CCooCAQw51sedpf/wlvp0EbWo/G/KTKTJaEpqDZC1hImkKZ1qLdr+64LeALPPb9NEqUnkIXrgAn+ufr4vlnSfHoP1PM8pCiqLgR7BesUz1VQ3Eqgr/ckQqihznbei4poFf8qE3wi7GysIuRtusGI4BHpk/RWlumYZPXSBZ9j+KuaQQajplo7bBAsf7b/w5Xw+7T1Ac/zEY3LUcpvGRU29yoec44ocDly07zT8unvh6J0qrm2UYblxxc0L0fGsn36dtNxnvwToVhGrYgZBWCf7vgG3vN2lY78I1TZXTO8J9yexR7Jn6GNC05HwhI0uM3iOsilYwmEc67oDiKC8dfKm42eEfVrYcrhZYE3rd8Q='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e9cd0-1d1e-7853-9bf5-b9d656e9767a-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'San Francisco'}, 'id': 'fc85df7a-455d-453e-a2cf-a6bfa2fabbee', 'type': 'tool_call'}], invalid_tool_ca